In [1]:
import tensorflow as tf
import numpy as np
import librosa
import json
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report

OUTPUT_DIR = Path("C:/Users/asalou/S7/stages7/project/augmentation/aug")
WAV_DIR    = Path("C:/Users/asalou/S7/stages7/project/wav_audio")
SEG_LEN    = 8000
N_MELS, N_FFT, HOP_MEL = 64, 512, 160

# ── LabelEncoder ──────────────────────────────────────────────
le = LabelEncoder()
le.fit(['bruit_ambiant', 'bruit_chasse', 'miction_active'])

# ── Charger modèle ────────────────────────────────────────────
model = tf.keras.models.load_model(f'{OUTPUT_DIR}/cnn2d_v2.keras')
print(f"Modèle chargé — input : {model.input_shape}")

# ── Mel spectrogram ───────────────────────────────────────────
def signal_to_melspec(segment):
    mel = librosa.feature.melspectrogram(
        y=segment.astype(float), sr=16000,
        n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_MEL, fmax=6000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mn, mx = mel_db.min(), mel_db.max()
    if mx - mn > 0:
        return ((mel_db - mn) / (mx - mn)).astype(np.float32)
    return np.zeros_like(mel_db, dtype=np.float32)

# ── Dataset calibration INT8 ──────────────────────────────────
print("Chargement segments calibration...")
cal_segments = []
for wav_file in sorted(WAV_DIR.glob("*.wav"))[:12]:
    y, _ = librosa.load(wav_file, sr=16000)
    pos  = 0
    while pos + SEG_LEN <= len(y) and len(cal_segments) < 300:
        seg = y[pos:pos+SEG_LEN]
        mel = signal_to_melspec(seg)
        cal_segments.append(mel[..., np.newaxis])
        pos += SEG_LEN
    if len(cal_segments) >= 300:
        break

cal_arr = np.array(cal_segments, dtype=np.float32)
print(f"Segments calibration : {len(cal_arr)}")

def representative_dataset():
    for i in range(len(cal_arr)):
        yield [cal_arr[i:i+1]]

# ════════════════════════════════════════════════════════════════
# VERSION 1 — Float32
# ════════════════════════════════════════════════════════════════
print("\nConversion Float32...")
conv_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_f32 = conv_f32.convert()
path_f32 = str(OUTPUT_DIR / 'cnn_v3_float32.tflite')
with open(path_f32, 'wb') as f:
    f.write(tflite_f32)
print(f" Float32 : {len(tflite_f32)/1024:.1f} KB")

# ════════════════════════════════════════════════════════════════
# VERSION 2 — Float16
# ════════════════════════════════════════════════════════════════
print("\nConversion Float16...")
conv_f16 = tf.lite.TFLiteConverter.from_keras_model(model)
conv_f16.optimizations = [tf.lite.Optimize.DEFAULT]
conv_f16.target_spec.supported_types = [tf.float16]
tflite_f16 = conv_f16.convert()
path_f16 = str(OUTPUT_DIR / 'cnn_v3_float16.tflite')
with open(path_f16, 'wb') as f:
    f.write(tflite_f16)
print(f" Float16 : {len(tflite_f16)/1024:.1f} KB")

# ════════════════════════════════════════════════════════════════
# VERSION 3 — INT8
# ════════════════════════════════════════════════════════════════
print("\nConversion INT8...")
conv_i8 = tf.lite.TFLiteConverter.from_keras_model(model)
conv_i8.optimizations             = [tf.lite.Optimize.DEFAULT]
conv_i8.representative_dataset    = representative_dataset
conv_i8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_i8.inference_input_type      = tf.int8
conv_i8.inference_output_type     = tf.int8
tflite_i8 = conv_i8.convert()
path_i8 = str(OUTPUT_DIR / 'cnn_v3_int8.tflite')
with open(path_i8, 'wb') as f:
    f.write(tflite_i8)
print(f" INT8 : {len(tflite_i8)/1024:.1f} KB")

Modèle chargé — input : (None, 64, 51, 1)
Chargement segments calibration...
Segments calibration : 300

Conversion Float32...
INFO:tensorflow:Assets written to: C:\Users\asalou\AppData\Local\Temp\tmp42l5dww1\assets


INFO:tensorflow:Assets written to: C:\Users\asalou\AppData\Local\Temp\tmp42l5dww1\assets


Saved artifact at 'C:\Users\asalou\AppData\Local\Temp\tmp42l5dww1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 51, 1), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1626027088832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027093760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027380400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027382512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027097104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027378112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027391312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027389552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027390432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027377056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  16260

INFO:tensorflow:Assets written to: C:\Users\asalou\AppData\Local\Temp\tmpogsjst7m\assets


Saved artifact at 'C:\Users\asalou\AppData\Local\Temp\tmpogsjst7m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 51, 1), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1626027088832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027093760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027380400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027382512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027097104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027378112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027391312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027389552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027390432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027377056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  16260

INFO:tensorflow:Assets written to: C:\Users\asalou\AppData\Local\Temp\tmple8yv5s7\assets


Saved artifact at 'C:\Users\asalou\AppData\Local\Temp\tmple8yv5s7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 51, 1), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1626027088832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027093760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027380400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027382512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027097104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027378112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027391312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027389552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027390432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1626027377056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  16260

C:\Users\asalou\S7\envs\stages7\lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 INT8 : 504.8 KB


In [2]:
# ── Charger labels + test set ─────────────────────────────────
with open(f'{OUTPUT_DIR}/labels_segments.json') as f:
    labels_check = json.load(f)

# Fusion silence → ambiant
for nom in labels_check:
    labels_check[nom] = [
        'bruit_ambiant' if l == 'silence' else l
        for l in labels_check[nom]
    ]

# Fichiers test — les mêmes qu'à l'entraînement
fichiers_test = ['Audio_1', 'Audio_7', 'Audio_5', 'Audio_10', 'Audio_17']

X_test_list, y_test_list = [], []
for nom in fichiers_test:
    wav_path = WAV_DIR / f"{nom}.wav"
    if not wav_path.exists():
        continue
    y_audio, _ = librosa.load(wav_path, sr=16000)
    labels_orig = labels_check[nom]
    n_orig = len(labels_orig)
    pos = 0
    while pos + SEG_LEN <= len(y_audio):
        seg     = y_audio[pos:pos+SEG_LEN]
        centre  = (pos + SEG_LEN//2) / 16000
        orig_id = min(int(centre / 0.5), n_orig-1)
        lbl     = labels_orig[orig_id]
        if lbl in ['bruit_ambiant','bruit_chasse','miction_active']:
            X_test_list.append(signal_to_melspec(seg)[..., np.newaxis])
            y_test_list.append(lbl)
        pos += SEG_LEN

X_test_mel = np.array(X_test_list, dtype=np.float32)
y_test_enc  = le.transform(y_test_list)
print(f"Test set : {len(X_test_mel)} segments")

# ── Fonction évaluation ───────────────────────────────────────
def evaluer_tflite(path, X_test, y_test, nom):
    interp = tf.lite.Interpreter(model_path=path)
    interp.allocate_tensors()
    inp = interp.get_input_details()
    out = interp.get_output_details()
    dtype = inp[0]['dtype']
    y_pred = []
    for i in range(len(X_test)):
        s = X_test[i:i+1]
        if dtype == np.int8:
            sc, zp = inp[0]['quantization']
            s = np.round(s / sc + zp).astype(np.int8)
        else:
            s = s.astype(np.float32)
        interp.set_tensor(inp[0]['index'], s)
        interp.invoke()
        y_pred.append(np.argmax(interp.get_tensor(out[0]['index'])))
    y_pred = np.array(y_pred)
    bal    = balanced_accuracy_score(y_test, y_pred)
    taille = Path(path).stat().st_size / 1024
    print(f"\n{'='*55}\n{nom}\n{'='*55}")
    print(f"Taille       : {taille:.1f} KB")
    print(f"Balanced Acc : {bal:.3f}")
    print(classification_report(
        y_test, y_pred,
        target_names=['bruit_ambiant','bruit_chasse','miction_active'],
        digits=3))
    return bal, taille

# Évaluer les 3 versions
bal_f32, t_f32 = evaluer_tflite(path_f32, X_test_mel, y_test_enc, "Float32")
bal_f16, t_f16 = evaluer_tflite(path_f16, X_test_mel, y_test_enc, "Float16")
bal_i8,  t_i8  = evaluer_tflite(path_i8,  X_test_mel, y_test_enc, "INT8")

# Tableau final
print("\n" + "="*70)
print("TABLEAU COMPARATIF — Choix quantification")
print("="*70)
print(f"{'Format':10s} | {'Taille':>10} | {'Bal.Acc':>9} | {'Perte':>8} | {'ATOMS3R':>15}")
print("-"*70)
print(f"{'Keras f32':10s} | {'référence':>10} | {0.857:9.3f} | {'—':>8} | {'Non':>15}")
print(f"{'Float32':10s} | {t_f32:>8.1f}KB | {bal_f32:9.3f} | {0.857-bal_f32:8.3f} | {'Non (trop lourd)':>15}")
print(f"{'Float16':10s} | {t_f16:>8.1f}KB | {bal_f16:9.3f} | {0.857-bal_f16:8.3f} | {'Partiel (RPi)':>15}")
print(f"{'INT8':10s} | {t_i8:>8.1f}KB | {bal_i8:9.3f} | {0.857-bal_i8:8.3f} | {'✅ ESP32-S3':>15}")
print("="*70)
print(f"\n→ Choix retenu : INT8 — meilleur compromis taille/précision ATOMS3R")

Test set : 373 segments


C:\Users\asalou\S7\envs\stages7\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Float32
Taille       : 1908.1 KB
Balanced Acc : 0.857
                precision    recall  f1-score   support

 bruit_ambiant      0.711     0.948     0.813       135
  bruit_chasse      0.821     0.920     0.868        50
miction_active      0.964     0.702     0.812       188

      accuracy                          0.820       373
     macro avg      0.832     0.857     0.831       373
  weighted avg      0.853     0.820     0.820       373



C:\Users\asalou\S7\envs\stages7\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Float16
Taille       : 958.7 KB
Balanced Acc : 0.857
                precision    recall  f1-score   support

 bruit_ambiant      0.711     0.948     0.813       135
  bruit_chasse      0.821     0.920     0.868        50
miction_active      0.964     0.702     0.812       188

      accuracy                          0.820       373
     macro avg      0.832     0.857     0.831       373
  weighted avg      0.853     0.820     0.820       373



C:\Users\asalou\S7\envs\stages7\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



INT8
Taille       : 504.8 KB
Balanced Acc : 0.848
                precision    recall  f1-score   support

 bruit_ambiant      0.692     0.948     0.800       135
  bruit_chasse      0.821     0.920     0.868        50
miction_active      0.962     0.676     0.794       188

      accuracy                          0.807       373
     macro avg      0.825     0.848     0.821       373
  weighted avg      0.845     0.807     0.806       373


TABLEAU COMPARATIF — Choix quantification
Format     |     Taille |   Bal.Acc |    Perte |         ATOMS3R
----------------------------------------------------------------------
Keras f32  |  référence |     0.857 |        — |             Non
Float32    |   1908.1KB |     0.857 |    0.000 | Non (trop lourd)
Float16    |    958.7KB |     0.857 |    0.000 |   Partiel (RPi)
INT8       |    504.8KB |     0.848 |    0.009 |      ✅ ESP32-S3

→ Choix retenu : INT8 — meilleur compromis taille/précision ATOMS3R


In [4]:
# Conversion INT8 → fichier C pour Arduino
with open(path_i8, 'rb') as f:
    data = f.read()

cc_path = str(OUTPUT_DIR / 'cnn_miction_model_v3.cc')
h_path  = str(OUTPUT_DIR / 'cnn_miction_model_v3.h')

with open(cc_path, 'w') as f:
    f.write('#include "cnn_miction_model_v3.h"\n\n')
    f.write('alignas(8) const unsigned char cnn_miction_model_v3[] = {\n  ')
    hex_vals = [f'0x{b:02x}' for b in data]
    lines    = [', '.join(hex_vals[i:i+12])
                for i in range(0, len(hex_vals), 12)]
    f.write(',\n  '.join(lines))
    f.write(f'\n}};\n\nunsigned int cnn_miction_model_v3_len = {len(data)};\n')

with open(h_path, 'w') as f:
    f.write('#ifndef CNN_MICTION_MODEL_V3_H\n')
    f.write('#define CNN_MICTION_MODEL_V3_H\n\n')
    f.write('extern const unsigned char cnn_miction_model_v3[];\n')
    f.write('extern unsigned int cnn_miction_model_v3_len;\n\n')
    f.write('#endif\n')

print(f" {cc_path}")
print(f"{h_path}")
print(f"Taille modèle : {len(data)/1024:.1f} KB")

 C:\Users\asalou\S7\stages7\project\augmentation\aug\cnn_miction_model_v3.cc
C:\Users\asalou\S7\stages7\project\augmentation\aug\cnn_miction_model_v3.h
Taille modèle : 504.8 KB
